# ATRIUM RSE Assessment Task

This notebook contains code and discussion comments to complete the assessment task for the position of Research Software Engineer on the ATRIUM project. The tasks focused on interacting with the SSH Open Marketplace and alongside possible solutions to the tasks I've included some thoughts/insights to the marketplace API which I hope are useful.

## Issues with the SSH Open Marketplace OpenAPI Specification

Whilst it is possible to interact with the marketplace API using simple HTTP requests it makes more sense
to use the [published OpenAPI specification](https://marketplace-api.sshopencloud.eu/v3/api-docs) to generate
a custom client library which can validate both the responses from the API and any new data we wish to
submit to the marketplace.

The code in this notebook generates a custom OpenAPI client library using the
[OpenAPI Generator from OpenAPITools](https://github.com/OpenAPITools/openapi-generator).

Unfortunately generating a client library which validates the API responses highlighted a few problems with the
currently published documentation for the marketplace.

### Fields Containing Dates

The biggest problem I've discovered relating to the OpenAPI documentation of the marketplace is that every field
containing a date appears to be incorrectly defined. For example, the `registrationDate` property of the `UserDto`
component is defined as follows:

```
"registrationDate":{
    "pattern":"yyyy-MM-dd'T'HH:mm:ssZ",
    "type":"string",
    "example":"2000-02-29T20:02:00+0000"
}
```

At first glance this looks sensible, and suggests that the date must conform to the `yyyy-MM-dd'T'HH:mm:ssZ`
format. The problem though, is that according to the OpenAPI specification the `pattern` field should be used
as a regular expression to validate the value of the string field. In this case, that means that only the
pattern string itself would, I beleive, be a valid value for this field. According to the OpenAPI specification
the correct way to specify dates is by using the  `format` field. In this case that would result in the following:

```
"registrationDate":{
    "format": "date-time",
    "type":"string",
    "example":"2000-02-29T20:02:00+0000"
}
```

The OpenAPI specification states that valid values for this field must conform to `date-time` as defined by
[RFC3339](https://datatracker.ietf.org/doc/html/rfc3339#section-5.6), which includes the expected
`yyyy-MM-dd'T'HH:mm:ssZ` format.

I assume this was missed because the marketplace OpenAPI description, as currently published, is syntactically
valid just not logically correct.

In order to generate a useable client library I've produced
[an updated version of the specification](https://raw.githubusercontent.com/greenwoodma/atrium-rse-assessment/refs/heads/main/api-docs.json)
which now correctly specifies dates.

### Required Fields

According to the [Metadata Guidelines](https://marketplace.sshopencloud.eu/contribute/metadata-guidelines)
for the marketplace both a `label` and `description` are mandatory for every resource type. The provided
OpenAPI description, however, states only that `label` is a required field meaning that a JSON object such as

```
{
    "label": "some label text"
}
```

would be a valid payload for creating a publication, dataset, training material, or workflow. At a minimum
(to match the guidelines) the `description` field should also be marked as required, but it is likely that
other fields should also be marked as required for many of the resource types, although without access to
the software behind the marketplace it is not possible to know which fields are truly required for the
creation of each resource type to succeed.

### Minimum/Maximum Values

There are a number of places within the API where the server applies minimum or maximum values to parameters
which are not documented. I encounted this when paging through search results as the `page` param has a minimum
value of 1 (not 0 as I would have assumed) and there is a maximum number of results `perPage` of 100. Neither
of these is documented and so can't be validated by clients prior to calling the API.

Minimum/Maximum values can be easily specified within the OpenAPI description. For example, documenting the
minimum value for the `page` param would look like the following:

```
{
  "name": "page",
  "in": "query",
  "required": false,
  "schema": {
    "type": "integer",
    "format": "int32",
    "minimum": 1
  }
}
```

Adding data such as this to the OpenAPI documentation makes it easier for developers exploring the API but
also alows client tools to validate there input which reduces load on the server by reducing invalid requests.

## Initial Environment Setup

This section of the notebook focuses on setting up the environment so that we can easily communicate with the marketplace.
None of the code in this section is specific to any of the assessment options.

We could have included the dependencies we need in a separate `requirements.txt` file but I've chosen instead to generate the file from within the notebook so that the dependencies we are using are made explicit and easy to view without having to switch to look at a separate file

In [1]:
%%writefile requirements.txt

# this is used to generate the custom client library to aid in
# accessing the marketplace REST API
openapi-generator-cli==7.21.0

# only needed for the data ananysis in the Option B task
pandas==3.0.2

Overwriting requirements.txt


We can now install the dependencies, generate the custom client library, and then install that into the notebook environment

In [2]:
# install all the dependencies we need for the rest of this notebook
%pip install -r requirements.txt

# generate a custom client library for working with the SSH Open Marketplace.
#
# note that we are using a modified copy of the OpenAPI description for the marketplace
# due to the issue around dates. Once the official version has been updated this
# command couild be updated to use https://marketplace-api.sshopencloud.eu/v3/api-docs
! openapi-generator-cli generate\
    -g python\
    -i https://raw.githubusercontent.com/greenwoodma/atrium-rse-assessment/refs/heads/main/api-docs.json\
    -o marketplace_client\
    --package-name marketplace_client

# install the custom client library into the environment ready for us to use
%pip install ./marketplace_client

Note: you may need to restart the kernel to use updated packages.
[main] INFO  o.o.codegen.DefaultGenerator - Generating with dryRun=false
[main] INFO  o.o.codegen.DefaultGenerator - OpenAPI Generator: python (client)
[main] INFO  o.o.codegen.DefaultGenerator - Generator 'python' is considered stable.
[main] INFO  o.o.c.l.AbstractPythonCodegen - Environment variable PYTHON_POST_PROCESS_FILE not defined so the Python code may not be properly formatted. To define it, try 'export PYTHON_POST_PROCESS_FILE="/usr/local/bin/yapf -i"' (Linux/Mac)
[main] INFO  o.o.c.l.AbstractPythonCodegen - NOTE: To enable file post-processing, 'enablePostProcessFile' must be set to `true` (--enable-post-process-file for CLI).
[main] WARN  o.o.codegen.DefaultCodegen - Unknown scheme `JWT` found in the HTTP security definition.
[main] WARN  o.o.codegen.utils.ModelUtils - Failed to get the schema name: null
[main] INFO  o.o.codegen.InlineModelResolver - Inline schema created as updateVocabulary_request. To have 

With the required dependencies installed we can now import and configure the custom client library

In [3]:
import marketplace_client

# By default the custom client library we created defaults to
# connecting to https://marketplace-api.sshopencloud.eu
# Should this need to be overriden a host parameter can be
# passed to the Configuration constructor. Note that this
# is where you would also specify authentication details etc.
configuration = marketplace_client.Configuration()

Before we start on the actual tasks let's also import some utility functions that just aid us in displaying the results

In [4]:
# this lets us nicely format output cells to show JSON or Markdown
from IPython.display import JSON, Markdown, display

## Option A – Item Type Conversion

The task is to take an item that has been created in the wrong category and map the data into a JSON object which could be used to re-create the object in the right category. I've chosen to map a Publication to a Dataset both as that was the example given in the task description but also as it seems the most likely real world mis-categorization. 

### Retrieve a Publication from the Marketplace

We start by specifying the persistent ID of the publication we want to convert. This can be found either from the `persistentId` field of a publication JSON record or from the end of the URL when viewed through the marketplace web application.

In [5]:
# As an example lets use the following publication
# https://marketplace.sshopencloud.eu/publication/cy3JNJ
# which is entitled:
#   The competitive esports physiological, affective, and video dataset
publication_id = "cy3JNJ"

Once we have the ID of a publication we can use the client library to retrieve it from the marketplace via the API

In [23]:

# Enter a context with a correctly configured instance of the API client
with marketplace_client.ApiClient(configuration) as api_client:
    
    # get the section of the API that handles publications
    # i.e. this gives access to everything in the spec tagged as
    # publication-controller which is essentially all the endpoints
    # under /api/publications/  
    publication_api = marketplace_client.PublicationControllerApi(api_client)
    
    # retrieve the specific publication we requested above
    # Note that the object returned is an instance of PublicationDto
    # as described in the OpenAPI specification document.
    publication = publication_api.get_publication(publication_id);

# display the retrieved JSON object so we can take a look at what
# we get back. Obviously not necessary in a real world system but
# useful here for debugging and sanity checking etc.
display(JSON(publication.to_dict(), expanded = True))

<IPython.core.display.JSON object>

### Convert the Publication into a Dataset

The `/api/datasets` endpoint used to create a new dataset within the marketplace expects us to POST a `DatasetCore` instance. Fortunately the client library makes it easy to create a `DatasetCore` instance by copying the values from a Python dict provided by the `PublicationDto` instance.

In [7]:

# If we want to create a new dataset from the publication then we need
# to complete an instance of DatasetCore that we could then POST to
# /api/datasets (using DatasetControllerAPI.create_dataset())
from marketplace_client.models import DatasetCore

# copying the data from the publication is easy as both objects support
# being converted to, or created from, a dict, so we convert the publication
# to a dict and then create a dataset from that dict.
#
# Because of the way this works inside the client library it recurses through
# nested objects copying the values that are needed. This coupled with the fact
# that the *Core components are all proper subsets of their *Dto counterparts
# means that while we ignore some information in the publication we should have
# everything we need to construct the dataset
dataset = DatasetCore.from_dict(publication.to_dict())

As an initial sanity check we can check which elements were not copied from the publicaiton into the new dataset instance

In [8]:
# turn the keys from both the publication and dataset into sets and then turn the
# difference between the two sets into a single string that we can format as
# a list in Markdown
Markdown("- "+"\n- ".join(publication.to_dict().keys() - dataset.to_dict().keys()))

- persistentId
- id
- status
- lastInfoUpdate
- informationContributor
- category

Given the dataset has not yet been created (we are just building the payload for the creation request) it makes sense that `id`, `persistentId`, `status`, and `lastInfoUpdate` are not copied across. Also we do not need to provide a `category` field as this will be automaticaly set to `dataset` by the marketplace.

That just leaves the `informationContributor` field. My understanding of this field is that it records the last person to edit the record. Given the dataset does not exist it makes sense that this field is not copied over, although it is unfortunate that creating a dataset in this way will sever the edit history at this point (assuming the original publication instance is deleted). One option would be to add the user into the list of contributors, but as the `informationContributor` is often a marketplace moderator, and not related to the creration of the original dataset, this does not seem a sensible option.

The [Metadata Guidelines](https://marketplace.sshopencloud.eu/contribute/metadata-guidelines) show that the main difference between a publication and a dataset relates to bibliographic metadata. All fields are recommended for publications but only `publisher` and `year` are recommended for datasets. It would be trivial to remove the other properties (such as `publication-type` etc.) but as providing them is not explicitly disallowed it makes more sense to leave them in place; someone thought they were important enough to provide originally and a moderator could always choose to remove them when accepting the new dataset record.

Unfortunately the [Metadata Guidelines](https://marketplace.sshopencloud.eu/contribute/metadata-guidelines) do not provide information on every field present in either the `PublicationDto` or `DatasetCore` components which does leave some ambiguity in the values of certain fields, especially `dateCreated` and `dateLastUpdated`. I assume that these both refer to the creation of the actual record within the marketplace and not to the creation or modified date of the publication or dataset being described. Assuming that is the case then it is rather odd to include them in the request to create a new dataset as I would assume the server would record the time it added the record. Given we are essentially converting one record to another, however, I'm tempted to provide both dates leaving the `dateCreated` as that given in the publication record but using the current time for the `dateLastUpdated` field.


In [9]:
from datetime import datetime

# update the dateLastUpdated prioperty to the current time
#
# note I've used a specific format pattern here to ensure the value matches the pattern given
# in the original OpenAPI docs for the marketplace (yyyy-MM-dd'T'HH:mm:ssZ) otherwise the
# default would be to include milliseconds as well. That would validate against the modified
# specification (i.e. it's a valid date as defined in RFC3339) in use with the client library
# but may cause an issue when submitted to the marketplace so this is safer.
dataset.date_last_updated = datetime.now().strftime("%Y-%m-%dT%H:%M:%S%Z")

One extra piece of metadata that would be potentially useful to add would be an explicit link back to the publication that the dataset was created from. The most obvious way of doing this would be with a `see-also` metadata property:

In [10]:
# adding a new property requires creating instances of the PropertyCore and
# PropertyTypeId components as defined in the marketplace OpenAPI spec so
# we import those models from the custom client library we generated
from marketplace_client.models import PropertyCore, PropertyTypeId

# append the new "see-also" property to the list of existing properties
dataset.properties.append(
    PropertyCore(
        type = PropertyTypeId(code = "see-also"),

        # I think a link to the marketplace web app makes more sense
        # here than a link to the API endpoint that would return JSON
        value = f"https://marketplace.sshopencloud.eu/publication/{publication_id}"
    )
)

### Produce the JSON Payload

With the mapping from publication to dataset completed we could now POST this to the `/api/datasets` endpoint to create the new entry in the marketplace. Given we don't want to do this though we will simply display the JSON payload instead

In [24]:
# converting the DatasetCore instance to a dict and then using the JSON
# IPython display function gives a nice human readbale form of the payload.
display(JSON(dataset.to_dict(), expanded = True))

<IPython.core.display.JSON object>

### Final Thoughts on the Task

The sections of the marketplace API used for this task are easy to use and intuitive. More detail of some properties, either in the guidelines or within the OpenAPI description, would be useful especially for fields such as `dateCreated` and `dateLastModified`. My feeling is that some of these fields are either already redundant (i.e. the marketplace will ignore their value) or should be removed and populated automatically (i.e. the marketplace should populate these values itself) -- being able to alter date based info used for tracking the edit history etc. does not seem like a sensible approach, although the danger of abuse is limited given you need to be authenticated in order to add a new dataset.

## Option B - Duplicate Detection

The aim of the task is to find duplicate entries in the marketplace, focusing specifically on the Tools & Services category.  Given the mixture of structured and free text information associated with each tool/service there are numerous ways we could look at finding duplicates. I've opted to look at both exact matching and a (simple) text similarity approach that can be used together to find a number of possible duplicate entries. The approach works in stages by first looking for exact matches and then using text similarity to rule out some of the more obvious false positives. This two stage approach keeps the simple text similarity approach feasible as it drastically reduces the number of comparisons needed.

### Retrieve the Tools/Services from the Marketplace

Whilst the task description suggests using a _meaningful sample_ of items from the tools & services category I decided that the dataset was small enough (the marketplace currently contains jusy 2,691 tools or services) that I could download information about all the tools as this would produce better results.

If you would prefer to retrieve less results then set the `pages` variable to the number of pages you wish to download. Each page of results contains 100 tools or services. Set the value to `sys.maxsize` to download all the pages of results.

In [12]:
import sys

# the number of pages (100 results per page) to download, or if set
# to sys.maxsize then download all pages of results
pages = sys.maxsize


_Note downloading the data takes a few minutes (approximately 8 on my home broadband connection), but feedback is provided during the process so it is easy to monitor. In a real world situation it is likely that duplication detection would be performed using more direct access to the database behind the marketplace. This would essentially remove the need for this more time consuming step._

In [13]:

# a list to hold the tool/service info we are about to download
tools = []

# Enter a context with a correctly configured instance of the API client
with marketplace_client.ApiClient(configuration) as api_client:
    
    # get the section of the API that handles tools and services
    # i.e. this gives access to everything in the spec tagged as
    # tool-controller which is essentially all the endpoints
    # under /api/tools-services/  
    tools_api = marketplace_client.ToolControllerApi(api_client)
    
    # for some strange reason the API counts pages from 1 not 0
    # so we want to start by getting page 1
    page = 1

    while page <= pages:
        # while there are still pages of results to collect...

        # give some feedback as this can take a while
        print(f"retrieving page {page} of {pages if pages != sys.maxsize else "?"}")
        
        # get the next page of tools from the marketplace
        response = tools_api.get_tools(page=page, perpage=100)
        
        # make sure we never try to download more pages of
        # results that there actually are
        if response.pages < pages:
            pages = response.pages
        
        for tool in response.tools:
            # convert each tool response we got back into a dict
            # and then add it to the list of tools we are collecting
            tools.append(tool.to_dict())
        
        # get ready to collect the next page of results
        page = page + 1 

# how many tools are there that we need to process?
print(f"Collected information on {len(tools)} tools or services")


retrieving page 1 of ?
retrieving page 2 of 27
retrieving page 3 of 27
retrieving page 4 of 27
retrieving page 5 of 27
retrieving page 6 of 27
retrieving page 7 of 27
retrieving page 8 of 27
retrieving page 9 of 27
retrieving page 10 of 27
retrieving page 11 of 27
retrieving page 12 of 27
retrieving page 13 of 27
retrieving page 14 of 27
retrieving page 15 of 27
retrieving page 16 of 27
retrieving page 17 of 27
retrieving page 18 of 27
retrieving page 19 of 27
retrieving page 20 of 27
retrieving page 21 of 27
retrieving page 22 of 27
retrieving page 23 of 27
retrieving page 24 of 27
retrieving page 25 of 27
retrieving page 26 of 27
retrieving page 27 of 27
Collected information on 2691 tools or services


### Find Potential Duplicates via `accessibleAt`

Having retrieved information about the tools and services we can move on to trying to find duplicates. As an initial first step I've opted to look for exact matches in the `accessibleAt` field of each tool. This field stores the URL from which the tool can be downloaded or accessed. Two or more tools that share a URL are likely to be duplicates or closely related.

The data we have downloaded so far is stored as a list of Python dict objetcs which is not the easiest of structures to work with. To enable us to aggreate/filter/sort the data the relevant elements are loaded into a pandas DataFrame.

In [14]:
# we are going to use the data handling and aggregation functions
# from DataFrame which is part of the pandas library so we need to
# import that here before we do anything else
import pandas as pd

# create a DataFrame where each row contains just the persistentId
# and accessiableAt data from the tools info we collected from the
# marketplace
df = pd.DataFrame(tools, columns=["persistentId", "accessibleAt"])


The data is now essentially in a two column table, where one column holds the `persistentId` of the tool and the second holds a list of zero or more `accessibleAt` URLs. What we really want is this flipped around so that for each URL we have a list of tools available from it.

In [15]:

# the accessibleAt filed is an array of zero or more URLs. This
# explodes the list so that each row is a URL and persistentId pair
df = df.explode("accessibleAt").reset_index(drop=True)

# if the accessibleAt array was empty then we end up with a row
# where the value is now null, and we can easily filter those out
df = df[df["accessibleAt"].notnull()]

# we now group the data based on the accessibleAt field so we end
# up with essentially the opposite of what we started with, where
# each row is now a URL and a list of persistentId values
df = df.groupby(['accessibleAt']).agg(pids=('persistentId', list)).reset_index()


With the data re-arranged we can look for possible duplicates by counting how many tools are associated with each URL, and filtering to keep only those URLs associated with more than one tool.

In [16]:

# add a new count column which is the number of persistentId values
df["count"] = df['pids'].str.len()

# filter the dataframe to only keep rows in which the same URL was
# associated with multiple tools
df = df[(df['count'] > 1)]

# display a summary (if for no other reason than it makes it clear
# that this cell has been run, given how fast it completes).
Markdown(f"""There are {df.shape[0]} URLs which are associated with two or more tools.
         These URLs relate to {df["count"].sum()} tools or services.""")

There are 64 URLs which are associated with two or more tools.
         These URLs relate to 176 tools or services.

Whilst a summary is useful what we really want is to look at the data and see if it seems sensible. Intuition tells me that URLs with fewer associated tools are more likely to relate to duplication; a URL associated with a large number of tools is likely to be hosting a suite of connected tools rather than there being a lot of duplication in the marketplace. As such I've sorted the URLs by increasing numbner of associated tools:

In [17]:

# sort the remaining values so we look at those URLs with the least
# tools first -- a URL with many tools is likely to be a suite of
# tools all available from the same URL rather than duplications
df = df.sort_values(by=['count'], ascending=True)

# create a new dataframe that essentially works as a mapping from the
# persistent ID of a tool to it's human readable label so that we can
# display these as part of the output
pidLabels = pd.DataFrame(tools, columns=["label", "persistentId"])

# given we are going to try a number of steps to find possible duplicates
# let's define a function we can use to pretty print each set so we can
# use it multiple times without having to simply copy-and-paste it across
# the notebook...

def displayDuplicates(duplicates: pd.DataFrame, labels: pd.DataFrame):
    """Displays each URL and the list of possibly duplicated tools
       as a Markdown list for nicer display in the notebook For each
       URL the outlook looks like

           <URL> is the URL for the following <X> possibly duplicate tools:
           - <label of tool 1>
           - <label of tool 2>

       Where the labels are also links back to the marketplace

       Parameters
       ----------

       duplicates: a pandas DataFrame where each row contains a URL and
            a list of persistent IDs (in the columns 'assesibleAt' and 'pids'
            respectively).
        labels: a pandas Dataframe that maps 'persistentId' fields to the
            'label' field for the tool

    """

    for row in duplicates.itertuples():
        # for each URL that we think is associated with duplicated tools,
        # build up and display a nicelty formated Markdown block to show
        # the URL and possible duplicated tools.
        display(Markdown(f"""[{row.accessibleAt}]({row.accessibleAt}) is the URL for the following {len(row.pids)} possibly duplicate tools:
        \n - {"\n - ".join([f"[{labels[labels["persistentId"] == p].iloc[0]["label"]}](https://marketplace.sshopencloud.eu/tool-or-service/{p})" for p in row.pids])}"""))


# now we have the function defined we can display the potential
# duplicates we think we have found from the initial analysis
displayDuplicates(df, pidLabels)

[http://alveo.edu.au/](http://alveo.edu.au/) is the URL for the following 2 possibly duplicate tools:
        
 - [Alveo](https://marketplace.sshopencloud.eu/tool-or-service/tAGITn)
 - [Alveo Virtual Laboratory](https://marketplace.sshopencloud.eu/tool-or-service/sJVV15)

[http://arm.lab.dariah.pl](http://arm.lab.dariah.pl) is the URL for the following 2 possibly duplicate tools:
        
 - [ARM: Automatic speech recognition](https://marketplace.sshopencloud.eu/tool-or-service/CJWXb5)
 - [ARMLatin: Medieval Latin document content acquisition](https://marketplace.sshopencloud.eu/tool-or-service/sArg4h)

[http://corpussearch.sourceforge.net/](http://corpussearch.sourceforge.net/) is the URL for the following 2 possibly duplicate tools:
        
 - [CorpusSearch](https://marketplace.sshopencloud.eu/tool-or-service/lvtnwT)
 - [CorpusSearch 2](https://marketplace.sshopencloud.eu/tool-or-service/GQE5se)

[http://kaskade.dwds.de/tei-tcf/](http://kaskade.dwds.de/tei-tcf/) is the URL for the following 2 possibly duplicate tools:
        
 - [TEI-TCF encoder+decoder](https://marketplace.sshopencloud.eu/tool-or-service/Q3Du9S)
 - [TEI↔TCF encoder+decoder](https://marketplace.sshopencloud.eu/tool-or-service/d9osio)

[http://mbostock.github.io/protovis/](http://mbostock.github.io/protovis/) is the URL for the following 2 possibly duplicate tools:
        
 - [Protovis](https://marketplace.sshopencloud.eu/tool-or-service/qDX6aU)
 - [Stanford Vis Group: Protovis](https://marketplace.sshopencloud.eu/tool-or-service/qUZguv)

...

[https://github.com/rbudac/McGill-Characterization-Process](https://github.com/rbudac/McGill-Characterization-Process) is the URL for the following 4 possibly duplicate tools:
        
 - [McGill Characterization Process - Alias Identifier](https://marketplace.sshopencloud.eu/tool-or-service/wvr0Vw)
 - [McGill Characterization Process - Calculate Character Centrality](https://marketplace.sshopencloud.eu/tool-or-service/CZy84j)
 - [McGill Characterization Process - Character Parser](https://marketplace.sshopencloud.eu/tool-or-service/gqZGVm)
 - [McGill Characterization Process - Collocate Parser](https://marketplace.sshopencloud.eu/tool-or-service/PQPtBr)

[https://music-put/lab.dariah.pl/en](https://music-put/lab.dariah.pl/en) is the URL for the following 5 possibly duplicate tools:
        
 - [GOST: Tool for the automatic generation of music for computer games](https://marketplace.sshopencloud.eu/tool-or-service/gDjdf7)
 - [MEIconvert: Symbolic music formats converter](https://marketplace.sshopencloud.eu/tool-or-service/Dy4y6u)
 - [MusicPUT: Online tools for automatic music transcription, timbre analysis, annotation for OMR, music generation for computer games and format conversion](https://marketplace.sshopencloud.eu/tool-or-service/cw1Ttk)
 - [ScoreScribe: Automatic audio-to-score transcription of monophonic music](https://marketplace.sshopencloud.eu/tool-or-service/Rj43Zx)
 - [Timbra: A tool for the analysis, parameterisation, visualisation and comparison of timbre](https://marketplace.sshopencloud.eu/tool-or-service/4JB2k3)

[https://analytics.umcs.pl/](https://analytics.umcs.pl/) is the URL for the following 6 possibly duplicate tools:
        
 - [3D fotogrametria: 3D models of any objects (photogrammetric method)](https://marketplace.sshopencloud.eu/tool-or-service/tLdtav)
 - [3D scanning of small and medium-sized objects by structured light method](https://marketplace.sshopencloud.eu/tool-or-service/EMAUcT)
 - [Geomagnetyka: Geomagnetic surveys](https://marketplace.sshopencloud.eu/tool-or-service/ITOQsy)
 - [GPR: Geophysical surveys using the GPR method](https://marketplace.sshopencloud.eu/tool-or-service/QIMWoe)
 - [Skan 2D: 2D scanning - digitisation](https://marketplace.sshopencloud.eu/tool-or-service/DwNIl5)
 - [UAV LiDAR](https://marketplace.sshopencloud.eu/tool-or-service/DNClMI)

[https://ws.clarin-pl.eu](https://ws.clarin-pl.eu) is the URL for the following 14 possibly duplicate tools:
        
 - [Inkluz](https://marketplace.sshopencloud.eu/tool-or-service/AAVvCx)
 - [Iobber](https://marketplace.sshopencloud.eu/tool-or-service/PLdNgx)
 - [Liner2](https://marketplace.sshopencloud.eu/tool-or-service/Sf46IH)
 - [MaltParser](https://marketplace.sshopencloud.eu/tool-or-service/9rngng)
 - [NER NLTK](https://marketplace.sshopencloud.eu/tool-or-service/8YmECL)
 - [ReSpa](https://marketplace.sshopencloud.eu/tool-or-service/ULF8GZ)
 - [Serel](https://marketplace.sshopencloud.eu/tool-or-service/GQQ79D)
 - [Spatial](https://marketplace.sshopencloud.eu/tool-or-service/ZAUXU6)
 - [Spejd](https://marketplace.sshopencloud.eu/tool-or-service/6Z4J7o)
 - [Summarize](https://marketplace.sshopencloud.eu/tool-or-service/YHMSNq)
 - [Tagger NLTK](https://marketplace.sshopencloud.eu/tool-or-service/cdKbsx)
 - [TermoPL](https://marketplace.sshopencloud.eu/tool-or-service/MHAZw5)
 - [WebSty](https://marketplace.sshopencloud.eu/tool-or-service/f1qFIg)
 - [WoSeDon](https://marketplace.sshopencloud.eu/tool-or-service/R7dpZ8)

[https://weblicht.sfs.uni-tuebingen.de/weblichtwiki/index.php/Main_Page](https://weblicht.sfs.uni-tuebingen.de/weblichtwiki/index.php/Main_Page) is the URL for the following 19 possibly duplicate tools:
        
 - [WebLicht Advanced Mode](https://marketplace.sshopencloud.eu/tool-or-service/nX5dWs)
 - [WebLicht All In One (DE)](https://marketplace.sshopencloud.eu/tool-or-service/wCgmIE)
 - [WebLicht All In One (NL)](https://marketplace.sshopencloud.eu/tool-or-service/tE9W1m)
 - [WebLicht Const Parsing DE](https://marketplace.sshopencloud.eu/tool-or-service/p7Dhoj)
 - [WebLicht Const Parsing EN](https://marketplace.sshopencloud.eu/tool-or-service/aYF4BV)
 - [WebLicht Dep Parsing DE](https://marketplace.sshopencloud.eu/tool-or-service/tlqboO)
 - [WebLicht Dep Parsing EN](https://marketplace.sshopencloud.eu/tool-or-service/bQzeZP)
 - [WebLicht Lemmas DE](https://marketplace.sshopencloud.eu/tool-or-service/QNXUuS)
 - [WebLicht Lemmas EN](https://marketplace.sshopencloud.eu/tool-or-service/EeAYop)
 - [WebLicht Morphology DE](https://marketplace.sshopencloud.eu/tool-or-service/Fic7R0)
 - [WebLicht Morphology EN](https://marketplace.sshopencloud.eu/tool-or-service/vayHx6)
 - [WebLicht NamedEntities DE](https://marketplace.sshopencloud.eu/tool-or-service/yxZvbH)
 - [WebLicht NamedEntities EN](https://marketplace.sshopencloud.eu/tool-or-service/Ou130i)
 - [WebLicht NamedEntities SL](https://marketplace.sshopencloud.eu/tool-or-service/cGWhKh)
 - [WebLicht POSTags Lemmas DE](https://marketplace.sshopencloud.eu/tool-or-service/8aKugO)
 - [WebLicht POSTags Lemmas EN](https://marketplace.sshopencloud.eu/tool-or-service/UVKako)
 - [WebLicht POSTags Lemmas FR](https://marketplace.sshopencloud.eu/tool-or-service/Z5BlVB)
 - [WebLicht POSTags Lemmas IT](https://marketplace.sshopencloud.eu/tool-or-service/MIO0FS)
 - [WebLicht Tokenization TUR](https://marketplace.sshopencloud.eu/tool-or-service/rBPdXP)

Looking down the list there are quite a few examples that, from the label alone, would appear to be duplicate entries (e.,g. Apache OpenNLP and Apache Open NLP). There are, however, a number of false positives including entries such as the WebLicht suite of tools and those from CLARIN-PL and McGill to name just a few.

### Reduce False Positives using Text Similarity

Finding duplicate entries using text similarity approaches can become computationally expensive. For example, even if we just focused on the label for each tool, we would need to check each label against every other label in the marketplace. There are efficent ways to do this, and it may be that the database behind the marketplace could do this quickly, however, for the 2,691 tools currently in the marketplace we would need to make 3,619,395 comparisons.

A possibly more sensible use of text similarity is to try and filter the set of possible duplicates we have found so far by removing those URLs where the labels of the associated tools are not particularly similar. The following does this using the [SequenceMatcher](https://docs.python.org/3/library/difflib.html) class from the standard Python difflib library. This is probably not the best similarity metric for this task but given time constraints is a useful starting point.

Specifically we are going to calculate the average similarity between the labels of all tools associated with a given URL and then use that score to further filter the potential duplicates.

In [18]:

# used for efficiently findoing all pairs of labels
from itertools import combinations

# used for simple/naive text similarity
import difflib

# the list will hold the scores for each row in the
# DataFrame, i.e. for each URL we are testing
scores = []

for row in df.itertuples():
    # for each URL get the set of labels for the associated tools
    selected = pidLabels[pidLabels["persistentId"].isin(row.pids)]["label"]
    
    # start with a score of 0
    score = 0.0
    
    # get all possible pairs of labels
    pairs = list(combinations(selected,2))
    
    for [p1, p2] in pairs: 
        # for each pair calculate the similarity of the labels
        # and the score to the running total.
        #
        # note we are assuming that case is not important so we
        # convert each label to lowercase before calculating the
        # similarity score
        score = score + difflib.SequenceMatcher(None, p1.lower(), p2.lower()).ratio()

    # compute the average score
    score = score / len(pairs)

    # store the score in the list
    scores.append(score)

# use the list of scores as a new column in the dataframe
df["score"] = scores


We can now filter the data to only keep those URLs where the associated tools have similar labels. The exact cutoff value for this is fairly arbitrary. I've opted to use 0.6 but you can change this value to see how well the similarity scores affects the final results.

In [19]:
# filter the dataframe to only keep rows in which the average similarity
# was above 0.6. This is fairly arbitray
filtered = df[(df['score'] > 0.6)]

With the filtering done we can sort the remaining URLs by the similarity score

In [20]:
# now we can sort the URLs by decending score so that the most likely duplicates
# appear at the start of the list
filtered = filtered.sort_values(by=['score'], ascending=False)

# display a summary (if for no other reason than it makes it clear
# that this cell has been run, given how fast it completes).
display(Markdown(f"""There are {filtered.shape[0]} URLs which are associated with two or more tools.
                 These URLs relate to {filtered["count"].sum()} tools or services."""))

There are 28 URLs which are associated with two or more tools.
                 These URLs relate to 60 tools or services.

And finally we can display the updated list of potential duplicates (using the utility function we wrote when we needed to do this before).

In [21]:
displayDuplicates(filtered, pidLabels)

[https://www.laurenceanthony.net/software/antconc/](https://www.laurenceanthony.net/software/antconc/) is the URL for the following 2 possibly duplicate tools:
        
 - [Antconc](https://marketplace.sshopencloud.eu/tool-or-service/doRnGH)
 - [AntConc](https://marketplace.sshopencloud.eu/tool-or-service/PRiGXG)

[https://webservices.cls.ru.nl/alpino](https://webservices.cls.ru.nl/alpino) is the URL for the following 2 possibly duplicate tools:
        
 - [Alpino-Webservice](https://marketplace.sshopencloud.eu/tool-or-service/3ar7oM)
 - [Alpino-Webservice](https://marketplace.sshopencloud.eu/tool-or-service/IN7Szz)

[https://interrogator.github.io/corpkit/](https://interrogator.github.io/corpkit/) is the URL for the following 2 possibly duplicate tools:
        
 - [Corpkit](https://marketplace.sshopencloud.eu/tool-or-service/74PRXu)
 - [CorpKit](https://marketplace.sshopencloud.eu/tool-or-service/8VsgT1)

[http://opennlp.apache.org/index.html](http://opennlp.apache.org/index.html) is the URL for the following 2 possibly duplicate tools:
        
 - [Apache Open NLP](https://marketplace.sshopencloud.eu/tool-or-service/rbo6MV)
 - [Apache OpenNLP](https://marketplace.sshopencloud.eu/tool-or-service/r43yWg)

[https://trolling.uit.no/](https://trolling.uit.no/) is the URL for the following 2 possibly duplicate tools:
        
 - [The Tromsø Repository of Language and Linguistics (TROLLing)](https://marketplace.sshopencloud.eu/tool-or-service/qHhN5d)
 - [Tromsø Repository of Language and Linguistics (TROLLing)](https://marketplace.sshopencloud.eu/tool-or-service/nQ1lnA)

...

[http://zil.ipipan.waw.pl/Concraft](http://zil.ipipan.waw.pl/Concraft) is the URL for the following 2 possibly duplicate tools:
        
 - [Concraft](https://marketplace.sshopencloud.eu/tool-or-service/djDOAb)
 - [Concraft-pl 2.0](https://marketplace.sshopencloud.eu/tool-or-service/GdDi7n)

[https://ct3.ortolang.fr/teiconvert/](https://ct3.ortolang.fr/teiconvert/) is the URL for the following 2 possibly duplicate tools:
        
 - [TEICONVERT](https://marketplace.sshopencloud.eu/tool-or-service/i5cny3)
 - [TEICORPO](https://marketplace.sshopencloud.eu/tool-or-service/HmiBe2)

[http://www.laurenceanthony.net/software/antwordprofiler/](http://www.laurenceanthony.net/software/antwordprofiler/) is the URL for the following 2 possibly duplicate tools:
        
 - [AntWordProfiler](https://marketplace.sshopencloud.eu/tool-or-service/vBrxsU)
 - [Laurence Anthony: AntWordProfiler](https://marketplace.sshopencloud.eu/tool-or-service/kfNHy8)

[http://ucrel.lancs.ac.uk/claws/](http://ucrel.lancs.ac.uk/claws/) is the URL for the following 2 possibly duplicate tools:
        
 - [CLAWS Part-of-Speech Tagger](https://marketplace.sshopencloud.eu/tool-or-service/Malu0k)
 - [CLAWS Tagger](https://marketplace.sshopencloud.eu/tool-or-service/NYujx4)

[https://lindat.cz/kontext](https://lindat.cz/kontext) is the URL for the following 2 possibly duplicate tools:
        
 - [KonText](https://marketplace.sshopencloud.eu/tool-or-service/RW7100)
 - [Kontext (LINDAT)](https://marketplace.sshopencloud.eu/tool-or-service/NYBAQ9)

When testing on the full dataset this has reduced the inital list by just over half, and from having examined the final list many of the suggestions do indeed appear to be valid duplicate entries in the marketplace which could be merged together.

### Final Thoughts on the Task

Finding duplicates across the marketplace is an interesting task and I can see further ways the task could be approached (looking at tools that share contributors for example). Ideally running a process like this over the full dataset is somthing that would be best done with more direct access to the underlying database: much of the data manipulation could probably have been done as a simple database query and would have the advantage of not having to download the full tool metadata either. Dependong on the database being used it may also be that more advanced text similarity options would be easy and efficient to access allowing for further experimentation.

One area where I could see the initial exact match approach being used though, would be as part of the workflow within the marketplace for registering a new tool or service. You can imagine that if the user proivided the `accessiableAt` URL and it was immediately checked to see what other tools were using the same URL then it may prevent duplicates being manually entered. Certainly showing the matches to the moderator tasked with reviewing the new item should allow many of these duplicates to be spotted quickly. I did wonder if it would be possible to show an example of this using the current search endpoint, but I couldn't find a reliable way of searching for exact matches on a given metadata field -- I can't find any documentation on what is allowed in a search query so I'm not sure which fields it is matching against or how.

If I'd had more time then I would have considered more complex text similarity approaches which may have performed better, but in this situation working just with the `label` field I'm not sure it would have helped considerably.

One final note on the API. The pagination of results works really well and I especially like that it tells you the number of result pages and not just the total number of results as that makes looping through the pages easier. One thing I did notices was that there is a mistake in the error response for `/api/tools-services`. If you set the `page` param to 0 then the response is

```
{
  "timestamp": "2026-04-23 09:47:54",
  "status": 400,
  "error": "Page index must not be less than zero"
}
```

The message should more properly read "Page index must not be less than one". Note that no where in the documentation (that I can find) does it state that the page numbers start at 1. Whilst starting at 1 for the UI makes sense, most people consuming the REST API would probably assume that the numbers count from zero so documenting this woud be helpful.